# Thí nghiệm Hybrid 1D-CNN + LSTM

Notebook này chạy lại pipeline trong `CNN_LSTM.py` với kiến trúc sequence-pooling mới: `LSTM(return_sequences=True) -> GlobalMaxPooling1D`.


In [2]:
from pathlib import Path
import os

START_DIR = Path.cwd().resolve()
if (START_DIR / 'cnn_lstm' / 'CNN_LSTM.py').exists():
    PROJECT_ROOT = START_DIR
    NOTEBOOK_DIR = PROJECT_ROOT / 'cnn_lstm'
elif (START_DIR / 'CNN_LSTM.py').exists() and START_DIR.name == 'cnn_lstm':
    NOTEBOOK_DIR = START_DIR
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError('Run this notebook from the repo root or the cnn_lstm directory.')

os.chdir(NOTEBOOK_DIR)

required = [
    NOTEBOOK_DIR / 'CNN_LSTM.py',
    PROJECT_ROOT / 'SQLInjection_XSS_MixDataset.1.0.0.csv',
    PROJECT_ROOT / 'csic_database.csv',
    PROJECT_ROOT / 'obfuscation_dataset_full_with_benign_shaped.xlsx',
]
for item in required:
    assert item.exists(), f'Missing file: {item}'

print('Start dir:', START_DIR)
print('Notebook dir:', NOTEBOOK_DIR)
print('Project root:', PROJECT_ROOT)
print('Found CNN_LSTM.py and all datasets.')


Start dir: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm
Notebook dir: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm
Project root: C:\Users\admin\Desktop\obfuscated-web-attack-detection
Found CNN_LSTM.py and all datasets.


## 1. Kiểm tra kiến trúc
Cell này chỉ dựng model và in shape, không huấn luyện. Kiến trúc mong đợi là CNN trích xuất đặc trưng cục bộ, LSTM trả sequence ở mọi timestep, rồi `GlobalMaxPooling1D` chọn tín hiệu ngữ cảnh mạnh nhất.


In [3]:
import importlib.util
import sys

module_path = NOTEBOOK_DIR / 'CNN_LSTM.py'
spec = importlib.util.spec_from_file_location('nckh_test_pipeline', module_path)
pipeline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pipeline
spec.loader.exec_module(pipeline)

demo_model = pipeline.build_model(vocab_size=191, max_len=1024, embedding_dim=64)
demo_model.summary()


Model: "Hybrid_1D_CNN_LSTM_Sequence_Pooling_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        12,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context_sequence (LSTM)    │ (None, 64, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_global_max_pool            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 258,881 (1011.25 KB)

 Trainable params: 258,881 (1011.25 KB)

 Non-trainable params: 0 (0.00 B)

## 2. Smoke test tùy chọn
Chạy trên mẫu nhỏ để kiểm tra môi trường. Không dùng kết quả này trong báo cáo.

In [3]:
%run CNN_LSTM.py --sample-size 3000 --obfu-sample-size 1000 --epochs 3 --batch-size 128 --output-dir artifacts_cnn_lstm_smoke

=== DATA PREPARED ===
Train           : 2,160
Val             : 240
Test            : 600
Obfuscated test : 1,000
Base p99 length : 981

=== TOKENIZER ===
Vocabulary size: 119
Tokenizer was fit on train payloads only.

=== INPUT SHAPES ===
X_train: (2160, 1024)
X_val  : (240, 1024)
X_test : (600, 1024)
X_obfu : (1000, 1024)
Class weights: {0: 1.4457831325301205, 1: 0.7643312101910829}


Model: "Hybrid_1D_CNN_LSTM_Sequence_Pooling_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │         7,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context_sequence (LSTM)    │ (None, 64, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_global_max_pool            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 254,273 (993.25 KB)

 Trainable params: 254,273 (993.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5024 - auc: 0.6931 - loss: 0.6491 - precision: 0.8565 - recall: 0.2765
Epoch 1: val_loss improved from None to 0.41899, saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras

Epoch 1: finished saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step - accuracy: 0.6412 - auc: 0.7879 - loss: 0.5797 - precision: 0.8789 - recall: 0.5237 - val_accuracy: 0.8542 - val_auc: 0.9033 - val_loss: 0.4190 - val_precision: 0.8910 - val_recall: 0.8854
Epoch 2/3
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step - accuracy: 0.8763 - auc: 0.9101 - loss: 0.3616 - precision: 0.9133 - recall: 0.8944
Epoch 2: val_loss improved from 0.41899 to 0.30095, saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras

Epoch 2: finished saving model to artifacts_cnn_lstm_smoke\best_hybrid_cnn_lstm.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 7s 367ms/step - accuracy: 0.8796 - auc: 0.9181 - loss: 0.3519 - 

## 3. Huấn luyện đầy đủ
Cell này chạy toàn bộ dữ liệu. EarlyStopping có thể dừng trước 50 epoch. Thời gian chạy trên CPU có thể kéo dài nhiều giờ.

In [4]:
%run CNN_LSTM.py --epochs 50 --batch-size 128 --output-dir artifacts_cnn_lstm_rerun

=== DATA PREPARED ===
Train           : 117,444
Val             : 13,050
Test            : 32,624
Obfuscated test : 150,000
Base p99 length : 977

=== TOKENIZER ===
Vocabulary size: 180
Tokenizer was fit on train payloads only.

=== INPUT SHAPES ===
X_train: (117444, 1024)
X_val  : (13050, 1024)
X_test : (32624, 1024)
X_obfu : (150000, 1024)
Class weights: {0: 1.3983093225383973, 1: 0.7783005738975997}


Model: "Hybrid_1D_CNN_LSTM_Sequence_Pooling_Web_Attack_Detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ char_embedding (Embedding)      │ (None, 1024, 64)       │        11,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k3 (Conv1D)                │ (None, 1024, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_1 (MaxPooling1D)           │ (None, 256, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_k5 (Conv1D)                │ (None, 256, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool_2 (MaxPooling1D)           │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_context_sequence (LSTM)    │ (None, 64, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_global_max_pool            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_classifier (Dense)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attack_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 258,177 (1008.50 KB)

 Trainable params: 258,177 (1008.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.9308 - auc: 0.9755 - loss: 0.1659 - precision: 0.9638 - recall: 0.9262
Epoch 1: val_loss improved from None to 0.05082, saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras

Epoch 1: finished saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras
918/918 ━━━━━━━━━━━━━━━━━━━━ 354s 381ms/step - accuracy: 0.9631 - auc: 0.9947 - loss: 0.0890 - precision: 0.9846 - recall: 0.9575 - val_accuracy: 0.9760 - val_auc: 0.9980 - val_loss: 0.0508 - val_precision: 0.9983 - val_recall: 0.9643
Epoch 2/50
918/918 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - accuracy: 0.9783 - auc: 0.9981 - loss: 0.0447 - precision: 0.9978 - recall: 0.9684
Epoch 2: val_loss improved from 0.05082 to 0.04167, saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras

Epoch 2: finished saving model to artifacts_cnn_lstm_rerun\best_hybrid_cnn_lstm.keras
918/918 ━━━━━━━━━━━━━━━━━━━━ 411s 412ms/step - accuracy: 0.9790 - auc: 0.998

In [ ]:
# Tune the decision threshold after training, using validation data only.
# Pipeline: train -> tune on validation -> evaluate normal/obfuscated test with the selected threshold.
import argparse
import json
import pickle
import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from tensorflow.keras.models import load_model

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cnn_lstm import CNN_LSTM as pipeline

ARTIFACT_DIR = Path('artifacts_cnn_lstm_rerun')
METADATA_PATH = ARTIFACT_DIR / 'metadata_and_results.json'
MODEL_PATH = ARTIFACT_DIR / 'last_hybrid_cnn_lstm.keras'
TOKENIZER_PATH = ARTIFACT_DIR / 'tokenizer.pkl'

with METADATA_PATH.open(encoding='utf-8') as f:
    metadata = json.load(f)

args = argparse.Namespace(
    kaggle_path=str(PROJECT_ROOT / 'SQLInjection_XSS_MixDataset.1.0.0.csv'),
    csic_path=str(PROJECT_ROOT / 'csic_database.csv'),
    obfuscation_path=str(PROJECT_ROOT / 'obfuscation_dataset_full_with_benign_shaped.xlsx'),
    test_size=0.2,
    val_size=0.1,
    seed=pipeline.SEED,
    sample_size=None,
    obfu_sample_size=None,
)
_, val_df, test_df, obfuscated_df, rebuilt_metadata = pipeline.build_datasets(args)

expected = metadata.get('splits', {})
# Only the validation/test splits must match the trained artifact.
# The obfuscated dataset is an external robustness test set and may be replaced without retraining.
for split_name, frame in [('val', val_df), ('test', test_df)]:
    expected_rows = expected.get(split_name, {}).get('rows')
    if expected_rows is not None and expected_rows != len(frame):
        raise ValueError(
            f"Rebuilt {split_name} has {len(frame)} rows, but metadata expects {expected_rows}. "
            'Check whether this artifact came from a sampled smoke run.'
        )
metadata.setdefault('splits', {})['obfuscated_test'] = pipeline.summarize(obfuscated_df)

max_len = metadata['model']['max_len']
with TOKENIZER_PATH.open('rb') as f:
    tokenizer = pickle.load(f)
model = load_model(MODEL_PATH, compile=False)

X_val = pipeline.vectorize(tokenizer, val_df['payload'], max_len)
X_test = pipeline.vectorize(tokenizer, test_df['payload'], max_len)
X_obfuscated = pipeline.vectorize(tokenizer, obfuscated_df['payload'], max_len)
y_val = val_df['label'].to_numpy(dtype=np.int32)
y_test = test_df['label'].to_numpy(dtype=np.int32)
y_obfuscated = obfuscated_df['label'].to_numpy(dtype=np.int32)


def metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    auc_roc = None
    if len(np.unique(y_true)) > 1:
        auc_roc = float(roc_auc_score(y_true, y_prob))
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'auc_roc': auc_roc,
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
        'classification_report': classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=['Normal (0)', 'Attack (1)'],
            digits=4,
            zero_division=0,
            output_dict=True,
        ),
    }


def print_result(title, result):
    auc_text = 'n/a' if result['auc_roc'] is None else f"{result['auc_roc']:.4f}"
    print(f"\n=== {title} @ threshold={result['threshold']:.2f} ===")
    print(f"Accuracy: {result['accuracy']:.4f}")
    print(f"AUC-ROC : {auc_text}")
    print('Confusion matrix:')
    print(np.array(result['confusion_matrix']))
    attack = result['classification_report']['Attack (1)']
    normal = result['classification_report']['Normal (0)']
    print(f"Normal recall : {normal['recall']:.4f}")
    print(f"Attack recall : {attack['recall']:.4f}")
    print(f"Attack F1     : {attack['f1-score']:.4f}")

val_prob = model.predict(X_val, batch_size=128).flatten()
test_prob = model.predict(X_test, batch_size=128).flatten()
obfuscated_prob = model.predict(X_obfuscated, batch_size=128).flatten()

thresholds = [round(float(x), 2) for x in np.arange(0.05, 0.96, 0.01)]
threshold_rows = [metrics_at_threshold(y_val, val_prob, t) for t in thresholds]
best = max(
    threshold_rows,
    key=lambda row: (
        row['classification_report']['Attack (1)']['f1-score'],
        row['classification_report']['Attack (1)']['recall'],
        row['classification_report']['Normal (0)']['recall'],
    ),
)
selected_threshold = best['threshold']

normal_default = metrics_at_threshold(y_test, test_prob, 0.5)
normal_tuned = metrics_at_threshold(y_test, test_prob, selected_threshold)
obfuscated_default = metrics_at_threshold(y_obfuscated, obfuscated_prob, 0.5)
obfuscated_tuned = metrics_at_threshold(y_obfuscated, obfuscated_prob, selected_threshold)

print(f'Selected threshold from validation: {selected_threshold:.2f}')
print_result('Normal test default', normal_default)
print_result('Normal test tuned', normal_tuned)
print_result('Obfuscated test default', obfuscated_default)
print_result('Obfuscated test tuned', obfuscated_tuned)

metadata['thresholding'] = {
    'mode': 'notebook_tuned_f1',
    'default_threshold': 0.5,
    'selected_threshold': selected_threshold,
    'selection_data': 'validation',
    'selection_rule': 'Maximize validation Attack (1) F1; tie-break by Attack (1) recall, then Normal (0) recall.',
}
metadata['threshold_search_val'] = threshold_rows
metadata.setdefault('evaluation', {})['normal_test_default_0_5'] = normal_default
metadata['evaluation']['normal_test_tuned_threshold'] = normal_tuned
metadata['evaluation']['test_default_0_5'] = normal_default
metadata['evaluation']['test_tuned_threshold'] = normal_tuned
metadata['evaluation']['obfuscated_test_default_0_5'] = obfuscated_default
metadata['evaluation']['obfuscated_test_tuned_threshold'] = obfuscated_tuned

with METADATA_PATH.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Saved tuned-threshold metrics to:', METADATA_PATH)


## 4. Đọc kết quả CNN-LSTM vừa chạy

In [5]:
import json

candidate_result_dirs = [
    NOTEBOOK_DIR / 'artifacts_cnn_lstm_rerun',
    NOTEBOOK_DIR / 'artifacts',
    NOTEBOOK_DIR / 'artifacts_cnn_lstm_smoke',
]
result_dir = next(
    (item for item in candidate_result_dirs if (item / 'metadata_and_results.json').exists()),
    None,
)
if result_dir is None:
    checked = '\n'.join(str(item) for item in candidate_result_dirs)
    raise FileNotFoundError('No CNN-LSTM metadata_and_results.json found. Checked:\n' + checked)

print('Reading CNN-LSTM results from:', result_dir)
with (result_dir / 'metadata_and_results.json').open(encoding='utf-8') as f:
    cnn_lstm_results = json.load(f)

print('Selected threshold:', cnn_lstm_results.get('thresholding', {}).get('selected_threshold', 'Run the threshold tuning cell first.'))
cnn_lstm_results['model'], cnn_lstm_results['evaluation']


Reading CNN-LSTM results from: C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm\artifacts_cnn_lstm_rerun


({'max_len': 1024,
  'embedding_dim': 64,
  'vocab_size': 180,
  'architecture': 'Embedding -> Conv1D(k3) -> MaxPool(4) -> Conv1D(k5) -> MaxPool(4) -> LSTM(128, return_sequences=True) -> GlobalMaxPooling1D -> Dense(64) -> Dropout -> Sigmoid',
  'architecture_note': 'Sequence pooling keeps the LSTM output at every timestep and pools the strongest activation, instead of relying only on the final LSTM state.',
  'class_weight': {'0': 1.3983093225383973, '1': 0.7783005738975997},
  'artifacts': {'best_model': 'artifacts_cnn_lstm_rerun\\best_hybrid_cnn_lstm.keras',
   'last_model': 'artifacts_cnn_lstm_rerun\\last_hybrid_cnn_lstm.keras',
   'tokenizer': 'artifacts_cnn_lstm_rerun\\tokenizer.pkl'}},
 {'test': {'accuracy': 0.9840608141245709,
   'auc_roc': 0.9990379168859136,
   'confusion_matrix': [[11567, 99], [421, 20537]],
   'classification_report': {'Normal (0)': {'precision': 0.9648815482148816,
     'recall': 0.9915138007886165,
     'f1-score': 0.9780164031453454,
     'support': 11666

## 5. So sánh CNN-only và CNN-LSTM
Hai model dùng cùng dữ liệu, tokenizer, seed, class weight và threshold 0.5.

In [15]:
import json
import pandas as pd

cnn_only_results_path = PROJECT_ROOT / 'cnn_only' / 'artifacts_new_preprocessing' / 'metadata_and_results.json'
if not cnn_only_results_path.exists():
    raise FileNotFoundError(
        'CNN-only results not found. Run cnn_only_experiment.ipynb first: '
        + str(cnn_only_results_path)
    )

with cnn_only_results_path.open(encoding='utf-8') as f:
    cnn_results = json.load(f)

cnn_eval = cnn_results['evaluation']
hybrid_eval = cnn_lstm_results['evaluation']
cnn_thresholding = cnn_results.get('thresholding', {})
hybrid_thresholding = cnn_lstm_results.get('thresholding', {})


def pick_result(evaluation, tuned_key, default_key, legacy_key):
    tuned = evaluation.get(tuned_key, evaluation.get(legacy_key))
    default = evaluation.get(default_key, evaluation.get(legacy_key))
    return default, tuned


def attack_metric(result, metric):
    return result['classification_report']['Attack (1)'][metric]


def obfu_recall(result):
    report = result['classification_report']
    if 'Attack (1)' in report:
        return report['Attack (1)']['recall']
    return report['1']['recall']


def missed_attacks(result):
    cm = result['confusion_matrix']
    return cm[1][0] if len(cm) > 1 else None


def estimate_cnn_lstm_params(model_meta):
    if model_meta.get('parameter_count') is not None:
        return model_meta['parameter_count']
    vocab_size = int(model_meta['vocab_size'])
    embedding_dim = int(model_meta.get('embedding_dim', 64))
    return int(
        vocab_size * embedding_dim
        + (3 * embedding_dim * 128) + 128
        + (5 * 128 * 128) + 128
        + 4 * ((128 + 128 + 1) * 128)
        + (128 * 64) + 64
        + (64 * 1) + 1
    )

cnn_default, cnn_tuned = pick_result(
    cnn_eval,
    tuned_key='normal_test_tuned_threshold',
    default_key='normal_test_default_0_5',
    legacy_key='normal_test',
)
hybrid_default, hybrid_tuned = pick_result(
    hybrid_eval,
    tuned_key='test_tuned_threshold',
    default_key='test_default_0_5',
    legacy_key='test',
)
cnn_obfu_default = cnn_eval.get('obfuscated_test_default_0_5', cnn_eval['obfuscated_test'])
cnn_obfu_tuned = cnn_eval.get('obfuscated_test_tuned_threshold', cnn_eval['obfuscated_test'])
hybrid_obfu_default = hybrid_eval.get('obfuscated_test_default_0_5', hybrid_eval['obfuscated_test'])
hybrid_obfu_tuned = hybrid_eval.get('obfuscated_test_tuned_threshold', hybrid_eval['obfuscated_test'])

comparison = pd.DataFrame([
    {
        'Model': 'CNN-only',
        'Params': cnn_results['cnn_only_model']['parameter_count'],
        'Selected Threshold': cnn_thresholding.get('selected_threshold', 0.5),
        'Default Accuracy': cnn_default['accuracy'],
        'Default Attack Recall': attack_metric(cnn_default, 'recall'),
        'Default Attack F1': attack_metric(cnn_default, 'f1-score'),
        'Tuned Accuracy': cnn_tuned['accuracy'],
        'Tuned AUC': cnn_tuned['auc_roc'],
        'Tuned Attack Recall': attack_metric(cnn_tuned, 'recall'),
        'Tuned Attack F1': attack_metric(cnn_tuned, 'f1-score'),
        'Default Obfuscated Accuracy': cnn_obfu_default['accuracy'],
        'Default Obfuscated Recall': obfu_recall(cnn_obfu_default),
        'Tuned Obfuscated Accuracy': cnn_obfu_tuned['accuracy'],
        'Tuned Obfuscated Recall': obfu_recall(cnn_obfu_tuned),
        'Tuned Obfuscated Missed': missed_attacks(cnn_obfu_tuned),
    },
    {
        'Model': 'CNN-LSTM rerun',
        'Params': estimate_cnn_lstm_params(cnn_lstm_results['model']),
        'Selected Threshold': hybrid_thresholding.get('selected_threshold', 0.5),
        'Default Accuracy': hybrid_default['accuracy'],
        'Default Attack Recall': attack_metric(hybrid_default, 'recall'),
        'Default Attack F1': attack_metric(hybrid_default, 'f1-score'),
        'Tuned Accuracy': hybrid_tuned['accuracy'],
        'Tuned AUC': hybrid_tuned['auc_roc'],
        'Tuned Attack Recall': attack_metric(hybrid_tuned, 'recall'),
        'Tuned Attack F1': attack_metric(hybrid_tuned, 'f1-score'),
        'Default Obfuscated Accuracy': hybrid_obfu_default['accuracy'],
        'Default Obfuscated Recall': obfu_recall(hybrid_obfu_default),
        'Tuned Obfuscated Accuracy': hybrid_obfu_tuned['accuracy'],
        'Tuned Obfuscated Recall': obfu_recall(hybrid_obfu_tuned),
        'Tuned Obfuscated Missed': missed_attacks(hybrid_obfu_tuned),
    },
])
comparison


,Model,Params,Selected Threshold,Default Accuracy,Default Attack Recall,Default Attack F1,Tuned Accuracy,Tuned AUC,Tuned Attack Recall,Tuned Attack F1,Default Obfuscated Accuracy,Default Obfuscated Recall,Tuned Obfuscated Accuracy,Tuned Obfuscated Recall,Tuned Obfuscated Missed
0,CNN-only,126593,0.55,0.983632,0.977526,0.987135,0.983386,0.998939,0.976620,0.986933,0.505097,0.999300,0.505937,0.999267,110
1,CNN-LSTM rerun,258177,0.47,0.984061,0.979912,0.987498,0.984153,0.999038,0.980294,0.987574,0.498980,0.997953,0.499030,0.998053,292


## Nguyên tắc kết luận

- Nếu CNN-LSTM tăng rõ Attack Recall hoặc Obfuscated Recall, LSTM có đóng góp thực nghiệm.
- Nếu kết quả gần ngang, ưu tiên CNN-only vì nhẹ hơn.
- Không kết luận chỉ dựa trên accuracy; cần xem False Negative, recall obfuscation, số tham số và thời gian chạy.